# v3: Controller-Guided Deep Research Agent

v2 的运行结果显示，很多样本在没有任何检索的情况下就进入 finalizer，导致平均搜索 query 过少、零检索失败较多。v3 改为 controller-guided：每题先由代码确定性执行 bootstrap search，再把证据交给模型继续推理和工具调用。

本 notebook 只定义方案和代码。本地不要执行、不要安装依赖、不要构建索引、不要启动或调用 vLLM。

## 1. 配置与导入

v3 使用 `get_v3_research_tool_specs_and_registry()`，在 v2 工具集基础上增加 `search_many`，并使用 `make_initial_search_queries()` 自动生成起手检索 query。

In [ ]:
from pathlib import Path
import json
import re
import sys
from typing import Any, Dict, List, Optional

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agent.dataset_utils import load_jsonl
from agent.eval import run_evaluation
from agent.tools import build_searcher, get_v3_research_tool_specs_and_registry, make_initial_search_queries
from agent.vllm_client import VLLMClient

VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen_auto"
API_KEY = "dummy"

BM25_INDEX_PATH = str(project_root / "indexes" / "browsecomp_plus_bm25.sqlite")
HARD50_PATH = str(project_root / "browsecomp_plus_hard50.jsonl")
SUBMISSION_PATH = str(project_root / "runs" / "v3_submission.jsonl")
EVAL_OUTPUT_PATH = str(project_root / "runs" / "v3_eval_results.jsonl")

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=BM25_INDEX_PATH)
tool_specs, tool_registry = get_v3_research_tool_specs_and_registry(
    searcher=searcher,
    k=8,
    snippet_max_chars=1400,
    document_window_chars=2200,
)


## 2. Prompt

v3 的 prompt 明确告诉模型：控制器已经提供了 bootstrap evidence，模型应在这些证据上继续查缺补漏，而不是从常识直接回答。

In [ ]:
SYSTEM_PROMPT = """
You are a controller-guided Deep Research Agent for BrowseComp-Plus.
Use only local tool evidence from the fixed BrowseComp-Plus corpus.

The controller will first provide bootstrap search results. Your job is to inspect them, identify promising docids,
open or search further when evidence is incomplete, and then answer with explicit support.

Tools:
- search(query): focused BM25 search.
- search_many(queries, per_query_k): run several searches and deduplicate results.
- get_document(docid): open a full document when necessary.
- get_document_window(docid, start, window_chars): inspect a bounded span.
- find_in_document(docid, keyword): locate a clue in a known document.
- collect_evidence_snippets(docids, keywords): gather final support snippets.

Rules:
1. Do not answer from memory.
2. Use the bootstrap results as evidence, but continue searching if key clues are unresolved.
3. Prefer precise phrases, quoted strings, years, names, titles, and unusual entities as search queries.
4. Before final answer, ground the answer in one or more docids or snippets when possible.
5. If evidence is weak, say so in Explanation and lower Confidence, but still return the best supported short answer.

Final answer format:
Explanation: <brief evidence-based reasoning>
Evidence: <docid: support summary; repeat if needed>
Exact Answer: <short final answer only>
Confidence: <0-100>%
""".strip()


FINALIZER_PROMPT = """
Finalize now. Do not call more tools.
Use only evidence already present in the conversation and compressed state.
Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()


## 3. 状态与工具执行辅助函数

v3 增加 controller tool call：这些调用会被写入 `messages`，因此提交轨迹仍可还原。答案抽取也会去掉 `<think>` 内容，避免把长推理文本当成 `predicted_answer`。

In [ ]:
def canonical_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "").strip().lower())


def truncate_text(text: str, max_chars: int) -> str:
    text = str(text or "")
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "... [truncated]"


def strip_thinking(text: str) -> str:
    text = str(text or "")
    text = re.sub(r"(?is)<think>.*?</think>\s*", "", text).strip()
    text = re.sub(r"(?is)<think>.*", "", text).strip()
    return text


def extract_exact_answer(text: str) -> str:
    original = str(text or "").strip()
    clean = strip_thinking(original)
    for candidate_text in (clean, original):
        match = re.search(r"(?im)^\s*Exact Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return match.group(1).strip()
        match = re.search(r"(?im)^\s*Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return match.group(1).strip()
    for pattern in (
        r"(?i)(?:the answer is|answer should be|most likely answer is)\s+[\"'“”]?([^\n\.;]+)",
        r"(?i)(?:therefore|thus),?\s+(?:the answer is)?\s*[\"'“”]?([^\n\.;]+)",
    ):
        match = re.search(pattern, original)
        if match:
            candidate = match.group(1).strip(" \"'“”")
            if 0 < len(candidate) <= 120:
                return candidate
    if clean and len(clean) <= 180:
        return clean
    return "Unable to determine"


def parse_tool_arguments(raw_arguments: Any) -> Dict[str, Any]:
    if isinstance(raw_arguments, dict):
        return raw_arguments
    if raw_arguments in (None, ""):
        return {}
    try:
        parsed = json.loads(raw_arguments)
    except json.JSONDecodeError:
        return {"_argument_parse_error": str(raw_arguments)}
    return parsed if isinstance(parsed, dict) else {"_argument_parse_error": str(raw_arguments)}


def normalize_tool_result(tool_name: str, result: Any, max_document_chars: int = 4500) -> Any:
    if tool_name == "get_document" and isinstance(result, dict):
        normalized = dict(result)
        if "text" in normalized:
            full_text = str(normalized.get("text", ""))
            normalized["text"] = truncate_text(full_text, max_document_chars)
            normalized["text_length"] = len(full_text)
            normalized["text_is_truncated"] = len(full_text) > max_document_chars
        return normalized
    return result


def execute_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name not in tool_registry:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": "unknown tool"}
    try:
        raw_result = tool_registry[tool_name](**arguments)
        return {
            "ok": True,
            "tool_name": tool_name,
            "arguments": arguments,
            "tool_result": normalize_tool_result(tool_name, raw_result),
        }
    except Exception as exc:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": repr(exc)}


def execute_tool_call(tool_call: Dict[str, Any]) -> Dict[str, Any]:
    function = tool_call.get("function", {}) or {}
    tool_name = function.get("name", "")
    arguments = parse_tool_arguments(function.get("arguments", "{}"))
    if "_argument_parse_error" in arguments:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": "invalid JSON arguments"}
    return execute_tool(tool_name, arguments)


def make_tool_message(tool_call_id: str, executed: Dict[str, Any]) -> Dict[str, Any]:
    content = executed.get("tool_result") if executed.get("ok") else executed
    return {
        "role": "tool",
        "tool_call_id": tool_call_id,
        "content": json.dumps(content, ensure_ascii=False),
    }


def docids_from_tool_result(tool_name: str, result: Any) -> List[str]:
    if tool_name in {"search", "search_many"} and isinstance(result, list):
        return [str(item.get("docid", "")) for item in result if isinstance(item, dict) and item.get("docid")]
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return [str(item.get("docid", "")) for item in result.get("snippets", []) if item.get("docid")]
    if isinstance(result, dict) and result.get("docid"):
        return [str(result.get("docid"))]
    return []


def initial_state(question: str) -> Dict[str, Any]:
    return {
        "question": question,
        "bootstrap_queries": [],
        "searched_queries": [],
        "searched_query_keys": [],
        "search_signatures": [],
        "seen_docids": [],
        "opened_docids": [],
        "evidence_table": [],
        "tool_events": [],
        "rounds": [],
        "low_gain_rounds": 0,
        "controller_actions": [],
    }


def summarize_tool_result(tool_name: str, result: Any) -> str:
    if tool_name in {"search", "search_many"}:
        return "docids=" + ", ".join(docids_from_tool_result(tool_name, result)[:10])
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return f"collected {len(result.get('snippets', []))} snippets"
    if tool_name == "find_in_document" and isinstance(result, dict):
        return f"{result.get('num_matches', 0)} matches for {result.get('keyword', '')} in {result.get('docid', '')}"
    if isinstance(result, dict) and result.get("docid"):
        return f"inspected docid={result.get('docid', '')}"
    return truncate_text(json.dumps(result, ensure_ascii=False), 300)


def update_state_from_execution(state: Dict[str, Any], executed: Dict[str, Any], round_id: int) -> int:
    tool_name = executed.get("tool_name", "")
    arguments = executed.get("arguments", {}) or {}
    result = executed.get("tool_result")
    new_signal = 0

    query_values = []
    if tool_name == "search":
        query_values = [arguments.get("query", "")]
    elif tool_name == "search_many":
        query_values = arguments.get("queries", []) or []
    for query in query_values:
        query = str(query or "")
        query_key = canonical_text(query)
        if query_key and query_key not in state["searched_query_keys"]:
            state["searched_query_keys"].append(query_key)
            state["searched_queries"].append(query)
            new_signal += 1

    if tool_name in {"search", "search_many"}:
        signature = ",".join(docids_from_tool_result(tool_name, result)[:10])
        if signature and signature not in state["search_signatures"]:
            state["search_signatures"].append(signature)
            new_signal += 1

    for docid in docids_from_tool_result(tool_name, result):
        if docid and docid not in state["seen_docids"]:
            state["seen_docids"].append(docid)
            new_signal += 1
        if tool_name in {"get_document", "get_document_window", "find_in_document"} and docid not in state["opened_docids"]:
            state["opened_docids"].append(docid)
            new_signal += 1

    if tool_name in {"find_in_document", "collect_evidence_snippets"}:
        state["evidence_table"].append(
            {
                "round_id": round_id,
                "tool_name": tool_name,
                "arguments": arguments,
                "summary": summarize_tool_result(tool_name, result),
            }
        )

    state["tool_events"].append(
        {
            "round_id": round_id,
            "tool_name": tool_name,
            "arguments": arguments,
            "ok": executed.get("ok", False),
            "summary": summarize_tool_result(tool_name, result),
        }
    )
    return new_signal


def public_state_summary(state: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "bootstrap_queries": state["bootstrap_queries"],
        "searched_queries": state["searched_queries"][-14:],
        "seen_docids": state["seen_docids"][-35:],
        "opened_docids": state["opened_docids"][-20:],
        "evidence_table": state["evidence_table"][-12:],
        "recent_tool_events": state["tool_events"][-14:],
        "low_gain_rounds": state["low_gain_rounds"],
    }


## 4. Controller Bootstrap 与 Agent Loop

关键变化：`bootstrap_research()` 在第一轮模型调用前写入一个 synthetic assistant tool call 和 tool result，保证每题至少有检索证据。若模型仍不调用工具且打开文档太少，控制器会自动打开候选文档窗口后再问一次。

In [ ]:
def append_controller_tool_call(
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    tool_name: str,
    arguments: Dict[str, Any],
    round_id: int,
    note: str,
) -> Dict[str, Any]:
    call_id = f"controller_{round_id}_{len(state['tool_events']) + 1}_{tool_name}"
    assistant_message = {
        "role": "assistant",
        "content": note,
        "tool_calls": [
            {
                "id": call_id,
                "type": "function",
                "function": {
                    "name": tool_name,
                    "arguments": json.dumps(arguments, ensure_ascii=False),
                },
            }
        ],
    }
    executed = execute_tool(tool_name, arguments)
    messages.append(assistant_message)
    messages.append(make_tool_message(call_id, executed))
    update_state_from_execution(state, executed, round_id)
    state["controller_actions"].append({"round_id": round_id, "tool_name": tool_name, "note": note})
    return executed


def bootstrap_research(
    question: str,
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    max_queries: int = 4,
    per_query_k: int = 5,
) -> None:
    queries = make_initial_search_queries(question, max_queries=max_queries)
    if not queries:
        queries = [question]
    state["bootstrap_queries"] = queries
    append_controller_tool_call(
        messages=messages,
        state=state,
        tool_name="search_many",
        arguments={"queries": queries, "per_query_k": per_query_k},
        round_id=0,
        note="Controller bootstrap: run initial BM25 searches before model reasoning.",
    )


def auto_open_candidate_window(
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    round_id: int,
    max_auto_opened: int = 2,
) -> bool:
    if len(state["opened_docids"]) >= max_auto_opened:
        return False
    for docid in state["seen_docids"]:
        if docid not in state["opened_docids"]:
            append_controller_tool_call(
                messages=messages,
                state=state,
                tool_name="get_document_window",
                arguments={"docid": docid, "start": 0, "window_chars": 2200},
                round_id=round_id,
                note="Controller recovery: inspect a top retrieved document before finalizing.",
            )
            return True
    return False


def build_model_messages(
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    recent_message_limit: int = 18,
) -> List[Dict[str, Any]]:
    state_message = {
        "role": "user",
        "content": (
            "Compressed research state. Continue from this state; do not repeat low-gain searches.\n"
            + json.dumps(public_state_summary(state), ensure_ascii=False, indent=2)
        ),
    }
    recent = messages[2:]
    if len(recent) > recent_message_limit:
        recent = recent[-recent_message_limit:]
        while recent and recent[0].get("role") == "tool":
            recent = recent[1:]
    return [messages[0], messages[1], state_message] + recent


def response_to_assistant_message(response: Dict[str, Any]) -> tuple[Dict[str, Any], List[Dict[str, Any]]]:
    message = response["choices"][0]["message"]
    assistant_message = {"role": "assistant", "content": message.get("content") or ""}
    tool_calls = message.get("tool_calls") or []
    if tool_calls:
        assistant_message["tool_calls"] = tool_calls
    return assistant_message, tool_calls


def finalize_answer(messages: List[Dict[str, Any]], state: Dict[str, Any], reason: str, max_tokens: int = 1100) -> str:
    final_instruction = {"role": "user", "content": reason + "\n\n" + FINALIZER_PROMPT}
    response = client.simple_chat(
        model=MODEL_NAME,
        messages=build_model_messages(messages, state) + [final_instruction],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    assistant_message, _ = response_to_assistant_message(response)
    messages.append(final_instruction)
    messages.append({"role": "assistant", "content": assistant_message.get("content", "")})
    return assistant_message.get("content", "")


def run_v3_agent(
    question: str,
    query_id: Optional[str] = None,
    max_rounds: int = 7,
    low_gain_round_limit: int = 2,
    max_tokens: int = 1300,
    recent_message_limit: int = 18,
) -> Dict[str, Any]:
    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    state = initial_state(question)
    bootstrap_research(question, messages, state)

    final_output = ""
    status = "max_rounds_reached"

    for round_id in range(1, max_rounds + 1):
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=build_model_messages(messages, state, recent_message_limit=recent_message_limit),
            temperature=0.0,
            max_tokens=max_tokens,
            tools=tool_specs,
            tool_choice="auto",
        )
        assistant_message, tool_calls = response_to_assistant_message(response)
        messages.append(assistant_message)
        state["rounds"].append(
            {
                "round_id": round_id,
                "assistant_content": truncate_text(assistant_message.get("content", ""), 500),
                "tool_call_count": len(tool_calls),
            }
        )

        if not tool_calls:
            if auto_open_candidate_window(messages, state, round_id=round_id):
                continue
            status = "completed_after_bootstrap"
            final_output = finalize_answer(
                messages,
                state,
                reason="The model stopped calling tools after bootstrap evidence. Produce the final answer.",
            )
            break

        round_signal = 0
        for tool_call in tool_calls:
            executed = execute_tool_call(tool_call)
            messages.append(make_tool_message(tool_call.get("id", ""), executed))
            round_signal += update_state_from_execution(state, executed, round_id)

        if round_signal == 0:
            state["low_gain_rounds"] += 1
        else:
            state["low_gain_rounds"] = 0

        if state["low_gain_rounds"] >= low_gain_round_limit:
            status = "stopped_low_gain_search"
            final_output = finalize_answer(
                messages,
                state,
                reason="Recent tool calls did not add new queries, documents, or evidence. Stop searching.",
            )
            break
    else:
        final_output = finalize_answer(
            messages,
            state,
            reason=f"Reached max_rounds={max_rounds}. Stop searching and answer from gathered evidence.",
        )

    return {
        "query_id": query_id,
        "status": status,
        "predicted_answer": extract_exact_answer(final_output),
        "final_output": final_output,
        "messages": messages,
        "state_summary": public_state_summary(state),
    }


## 5. Submission 与评测

v2 中出现 gold/pred 完全相同但 eval 被截断判错的情况。v3 的 `evaluate_submission()` 将 `max_tokens` 提高到 1024，降低 reasoning 模型没输出 `Judgment:` 就被截断的风险。

In [ ]:
def build_submission_record(row: Dict[str, Any], **agent_kwargs: Any) -> Dict[str, Any]:
    result = run_v3_agent(
        question=row["query"],
        query_id=str(row.get("query_id", "")),
        **agent_kwargs,
    )
    return {
        "query_id": str(row.get("query_id", "")),
        "predicted_answer": result["predicted_answer"],
        "status": result["status"],
        "messages": result["messages"],
        "state_summary": result["state_summary"],
    }


def generate_submission(
    dataset_path: str = HARD50_PATH,
    output_path: str = SUBMISSION_PATH,
    limit: Optional[int] = 50,
    **agent_kwargs: Any,
) -> List[Dict[str, Any]]:
    rows = load_jsonl(dataset_path, limit=limit)
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    records = []
    with output_file.open("w", encoding="utf-8") as fout:
        for index, row in enumerate(rows, start=1):
            print(f"[{index}/{len(rows)}] query_id={row.get('query_id', '')}")
            record = build_submission_record(row, **agent_kwargs)
            records.append(record)
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records to {output_path}")
    return records


def evaluate_submission(
    submission_path: str = SUBMISSION_PATH,
    dataset_path: str = HARD50_PATH,
    output_path: str = EVAL_OUTPUT_PATH,
):
    return run_evaluation(
        submission_path=submission_path,
        dataset_path=dataset_path,
        model_name=MODEL_NAME,
        base_url=VLLM_BASE_URL,
        api_key=API_KEY,
        output_path=output_path,
        max_tokens=1024,
        verbose=True,
    )


## 6. 服务器运行入口

本地不要执行。到服务器确认 vLLM 和 BM25 索引已就绪后，再取消注释运行。

```python
# records = generate_submission(limit=50, max_rounds=7, low_gain_round_limit=2)
# summary, details = evaluate_submission()
```